# Customer Shopping Behavior Analysis
## Data Cleaning & MySQL Integration

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv("customer_shopping_behavior.csv")
df.head()

In [ ]:
# Dataset Overview
df.info()
df.describe(include='all')

In [ ]:
# Missing Values
df.isnull().sum()

In [ ]:
# Fill missing review ratings with median (if applicable)
if 'review_rating' in df.columns:
    df['review_rating'] = df['review_rating'].fillna(df['review_rating'].median())


In [ ]:
# Rename columns to snake_case
df.columns = (
    df.columns.str.strip()
              .str.lower()
              .str.replace(" ","_",regex=False)
)
df.columns

In [ ]:
# Feature Engineering - Age Group
if 'age' in df.columns:
    df['age_group'] = pd.cut(
        df['age'],
        bins=[0,25,40,60,100],
        labels=['18-25','26-40','41-60','60+']
    )

# Purchase Frequency
if 'previous_purchases' in df.columns:
    df['purchase_frequency_days'] = np.where(
        df['previous_purchases']>0,
        365/df['previous_purchases'],
        np.nan
    )

# Drop redundant column
if 'promo_code_used' in df.columns:
    df.drop(columns=['promo_code_used'], inplace=True)

df.head()

## Connect to MySQL

In [ ]:
# Install once if required
# !pip install pymysql sqlalchemy

In [ ]:
from sqlalchemy import create_engine

username = "root"
password = "PASSWORD"
host = "localhost"
port = "3306"
database = "customer_behavior"

engine = create_engine(
    f"mysql+pymysql://{username}:{password}@{host}:{port}/{database}"
)

df.to_sql(
    name="customer",
    con=engine,
    if_exists="replace",
    index=False
)

print("Data uploaded successfully to MySQL.")

In [ ]:
# Verify Upload
query = "SELECT * FROM customer LIMIT 5;"
pd.read_sql(query, engine)